In [1]:
import importlib.metadata

try:
    if importlib.metadata.version("numpy") is None:
        _original_version = importlib.metadata.version
        importlib.metadata.version = (
            lambda name: "1.26.4"
            if name.lower() == "numpy"
            else _original_version(name)
        )
except Exception:
    _original_version = importlib.metadata.version
    importlib.metadata.version = (
        lambda name: "1.26.4"
        if name.lower() == "numpy"
        else _original_version(name)
    )

from pathlib import Path
import datetime
import yaml
import numpy as np
import weldx

from weldx import WeldxFile, Q_, GmawProcess
from weldx.core import TimeSeries


# ============================================================
# FILES AND PROJECT METADATA
# ============================================================
CORE_FILE = "project_core.asdf"
CONFIG_FILE = "config.yaml"

print(f"Initializing WelDX core file: {CORE_FILE}")

_weldx_version = getattr(weldx, "__version__", "unknown")
_numpy_version = np.__version__
_created_at = datetime.datetime.now().isoformat(timespec="seconds")


# ============================================================
# LOAD EXISTING CONFIGURATION
# ============================================================
config_path = Path(CONFIG_FILE)

if not config_path.exists():
    raise FileNotFoundError(
        f"⚠ {CONFIG_FILE} not found. "
        "Place config.yaml in the same folder as this notebook."
    )

with config_path.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

if not isinstance(config, dict):
    raise ValueError(
        f"{CONFIG_FILE} must contain a YAML mapping at the top level."
    )

if "gmaw_process" not in config:
    raise KeyError(
        "The existing config.yaml must contain a 'gmaw_process' section. "
        "Its voltage, current, travel speed and wire-feed values are retained."
    )

# Preserve these sections exactly as provided by the original project.
_gmaw_config_unchanged = config["gmaw_process"]
_monte_carlo_config_unchanged = config.get("monte_carlo")

_existing_material = config.get("material", {})
ambient_temperature_config = _existing_material.get(
    "ambient_temperature",
    {
        "value": 20.0,
        "unit": "degC",
    },
)


# ============================================================
# MATERIAL: ALUMINIUM 6061-T6
# ============================================================
config["material"] = {
    "common_name": "Aluminium 6061-T6",
    "standard": "AL 6061-T6 / ASTM B209",
    "density": {
        "value": 2700.0,
        "unit": "kg/m^3",
    },
    "thermal_conductivity": {
        "value": 167.0,
        "unit": "W/(m*K)",
    },
    "specific_heat": {
        "value": 896.0,
        "unit": "J/(kg*K)",
    },
    "melting_point": {
        "value": 652.0,
        "unit": "degC",
    },
    "solidus_temperature": {
        "value": 582.0,
        "unit": "degC",
    },
    "ambient_temperature": ambient_temperature_config,
    "data_source": (
        "Representative room-temperature thermophysical properties "
        "for wrought Aluminium 6061-T6."
    ),
}


# ============================================================
# CORRECTED DOUBLE-FILLET LAP-JOINT GEOMETRY
# ============================================================
# Correct coordinate convention:
#   x = plate length and weld-travel direction
#   y = plate width and overlap-normal direction
#   z = plate thickness direction
#
# Lower plate:
#   x = 0 to 200 mm
#   y = 0 to 80 mm
#   z = 0 to 4 mm
#
# Upper plate:
#   x = 0 to 200 mm
#   y = 55 to 135 mm
#   z = 4 to 8 mm
#
# Overlap:
#   y = 55 to 80 mm, giving 25 mm overlap
#
# Weld 1:
#   y = 55 mm, x = 0 to 200 mm
#
# Weld 2:
#   y = 80 mm, x = 0 to 200 mm

plate_length_mm = 200.0
plate_width_mm = 80.0
plate_thickness_mm = 4.0
overlap_mm = 25.0
fillet_leg_mm = 4.0

assembly_length_mm = plate_length_mm
assembly_width_mm = (2.0 * plate_width_mm) - overlap_mm

lower_y_start_mm = 0.0
lower_y_end_mm = plate_width_mm

upper_y_start_mm = plate_width_mm - overlap_mm
upper_y_end_mm = upper_y_start_mm + plate_width_mm

weld_line_y_mm = [
    upper_y_start_mm,
    lower_y_end_mm,
]

weld_length_mm = plate_length_mm

thermocouple_distance_mm = [
    2.0,
    5.0,
    10.0,
]

# Thermocouples are placed at the middle of the 200 mm weld length.
# Their distances are measured normally away from Weld 1 on the lower plate.
thermocouple_x_mm = [
    plate_length_mm / 2.0,
] * 3

thermocouple_y_mm = [
    weld_line_y_mm[0] - distance
    for distance in thermocouple_distance_mm
]

thermocouple_z_mm = [
    plate_thickness_mm,
] * 3

weld_path_1_start_mm = [
    0.0,
    weld_line_y_mm[0],
    plate_thickness_mm,
]

weld_path_1_end_mm = [
    plate_length_mm,
    weld_line_y_mm[0],
    plate_thickness_mm,
]

weld_path_2_start_mm = [
    0.0,
    weld_line_y_mm[1],
    plate_thickness_mm,
]

weld_path_2_end_mm = [
    plate_length_mm,
    weld_line_y_mm[1],
    plate_thickness_mm,
]

config["geometry"] = {
    "joint_type": "Lap joint",
    "joint_configuration": (
        "Double fillet weld on both longitudinal edges of the overlap"
    ),
    "data_source": "User-specified corrected lap-joint geometry",

    "coordinate_system": {
        "x": "plate-length and weld-travel direction",
        "y": "plate-width and overlap-normal direction",
        "z": "plate-thickness direction",
    },

    "plate_length": {
        "value": plate_length_mm,
        "unit": "mm",
    },

    "plate_width": {
        "value": plate_width_mm,
        "unit": "mm",
    },

    "plate_thickness": {
        "value": plate_thickness_mm,
        "unit": "mm",
    },

    "assembly_length": {
        "value": assembly_length_mm,
        "unit": "mm",
    },

    "assembly_width": {
        "value": assembly_width_mm,
        "unit": "mm",
    },

    "lower_plate": {
        "length": {
            "value": plate_length_mm,
            "unit": "mm",
        },
        "width": {
            "value": plate_width_mm,
            "unit": "mm",
        },
        "thickness": {
            "value": plate_thickness_mm,
            "unit": "mm",
        },
        "x_range": {
            "values": [0.0, plate_length_mm],
            "unit": "mm",
        },
        "y_range": {
            "values": [lower_y_start_mm, lower_y_end_mm],
            "unit": "mm",
        },
        "z_range": {
            "values": [0.0, plate_thickness_mm],
            "unit": "mm",
        },
    },

    "upper_plate": {
        "length": {
            "value": plate_length_mm,
            "unit": "mm",
        },
        "width": {
            "value": plate_width_mm,
            "unit": "mm",
        },
        "thickness": {
            "value": plate_thickness_mm,
            "unit": "mm",
        },
        "x_range": {
            "values": [0.0, plate_length_mm],
            "unit": "mm",
        },
        "y_range": {
            "values": [upper_y_start_mm, upper_y_end_mm],
            "unit": "mm",
        },
        "z_range": {
            "values": [plate_thickness_mm, 2.0 * plate_thickness_mm],
            "unit": "mm",
        },
    },

    "overlap": {
        "value": overlap_mm,
        "unit": "mm",
    },

    "overlap_y_range": {
        "values": [upper_y_start_mm, lower_y_end_mm],
        "unit": "mm",
    },

    "fillet_leg": {
        "value": fillet_leg_mm,
        "unit": "mm",
    },

    "weld_count": 2,

    "weld_sides": [
        "first longitudinal edge of overlap",
        "second longitudinal edge of overlap",
    ],

    "weld_length": {
        "value": weld_length_mm,
        "unit": "mm",
    },

    "weld_line_y_positions": {
        "values": weld_line_y_mm,
        "unit": "mm",
    },

    "weld_travel_direction": "+x",

    "weld_path_1_start": {
        "values": weld_path_1_start_mm,
        "unit": "mm",
    },

    "weld_path_1_end": {
        "values": weld_path_1_end_mm,
        "unit": "mm",
    },

    "weld_path_2_start": {
        "values": weld_path_2_start_mm,
        "unit": "mm",
    },

    "weld_path_2_end": {
        "values": weld_path_2_end_mm,
        "unit": "mm",
    },

    "sensor_distances": {
        "values": thermocouple_distance_mm,
        "unit": "mm",
        "reference": "normal distance from Weld 1 centreline",
    },

    "sensor_labels": [
        "TC1",
        "TC2",
        "TC3",
    ],

    "thermocouple_reference_weld": (
        "Weld 1 at y = 55 mm"
    ),

    "thermocouple_x_positions": {
        "values": thermocouple_x_mm,
        "unit": "mm",
    },

    "thermocouple_y_positions": {
        "values": thermocouple_y_mm,
        "unit": "mm",
    },

    "thermocouple_z_positions": {
        "values": thermocouple_z_mm,
        "unit": "mm",
    },
}

# Restore the existing GMAW and Monte Carlo settings unchanged.
config["gmaw_process"] = _gmaw_config_unchanged

if _monte_carlo_config_unchanged is not None:
    config["monte_carlo"] = _monte_carlo_config_unchanged

with config_path.open("w", encoding="utf-8") as file:
    yaml.safe_dump(
        config,
        file,
        sort_keys=False,
        allow_unicode=True,
    )


print(f"Loaded and updated configuration: {CONFIG_FILE}")
print(f"  Material       : {config['material']['common_name']}")
print("  Process        : GMAW")
print(
    f"  Voltage/Current: "
    f"{config['gmaw_process']['voltage']['value']} "
    f"{config['gmaw_process']['voltage']['unit']} / "
    f"{config['gmaw_process']['current']['value']} "
    f"{config['gmaw_process']['current']['unit']}"
)
print(
    f"  Wire feed      : "
    f"{config['gmaw_process']['wire_feedrate']['value']} "
    f"{config['gmaw_process']['wire_feedrate']['unit']}"
)
print(f"  Plate size     : 200 × 80 × 4 mm")
print(f"  Overlap        : 25 mm across plate width")
print(f"  Weld length    : 200 mm per fillet weld")
print(f"  Weld direction : +x")


# ============================================================
# ALUMINIUM 6061 MATERIAL OBJECT
# ============================================================
_mat = config["material"]

material = {
    "common_name": _mat["common_name"],
    "standard": _mat["standard"],
    "density": Q_(
        _mat["density"]["value"],
        _mat["density"]["unit"],
    ),
    "thermal_conductivity": Q_(
        _mat["thermal_conductivity"]["value"],
        _mat["thermal_conductivity"]["unit"],
    ),
    "specific_heat": Q_(
        _mat["specific_heat"]["value"],
        _mat["specific_heat"]["unit"],
    ),
    "melting_point": Q_(
        _mat["melting_point"]["value"],
        _mat["melting_point"]["unit"],
    ),
    "solidus_temperature": Q_(
        _mat["solidus_temperature"]["value"],
        _mat["solidus_temperature"]["unit"],
    ),
    "ambient_temperature": Q_(
        _mat["ambient_temperature"]["value"],
        _mat["ambient_temperature"]["unit"],
    ),
    "data_source": _mat["data_source"],
}

material["thermal_diffusivity"] = (
    material["thermal_conductivity"]
    / (
        material["density"]
        * material["specific_heat"]
    )
).to("m^2/s")

print("\n=== Aluminium 6061-T6 Material Properties ===")

for key, value in material.items():
    print(f"  {key:30s}: {value}")


# ============================================================
# GMAW PROCESS — EXISTING VALUES RETAINED
# ============================================================
_proc = config["gmaw_process"]

voltage_val = Q_(
    _proc["voltage"]["value"],
    _proc["voltage"]["unit"],
)

current_val = Q_(
    _proc["current"]["value"],
    _proc["current"]["unit"],
)

speed_val = Q_(
    _proc["travel_speed"]["value"],
    _proc["travel_speed"]["unit"],
)

wire_feedrate_val = Q_(
    _proc["wire_feedrate"]["value"],
    _proc["wire_feedrate"]["unit"],
)

efficiency = _proc["efficiency"]

ts_voltage = TimeSeries(voltage_val)
ts_current = TimeSeries(current_val)
ts_wire_feedrate = TimeSeries(wire_feedrate_val)

gmaw_process = GmawProcess(
    base_process=_proc["base_process"],
    manufacturer=_proc["manufacturer"],
    power_source=_proc["power_source"],
    parameters={
        "wire_feedrate": ts_wire_feedrate,
        "voltage": ts_voltage,
        "current": ts_current,
    },
    tag="GMAW",
    meta={
        "shielding_gas": _proc["shielding_gas"],
        "efficiency": efficiency,
    },
)

power_nom = (
    voltage_val
    * current_val
).to("W")

net_power_nom = (
    power_nom
    * efficiency
)

heat_input_nom = (
    net_power_nom
    / speed_val
).to("J/m")

process_scalars = {
    "process_name": "GMAW (MIG/MAG)",
    "data_source": _proc["data_source"],
    "voltage": voltage_val,
    "current": current_val,
    "speed": speed_val,
    "efficiency": Q_(
        efficiency,
        "dimensionless",
    ),
    "shielding_gas": _proc["shielding_gas"],
    "power": power_nom,
    "net_power": net_power_nom,
    "heat_input": heat_input_nom,
    "wire_feedrate": wire_feedrate_val,
}

print("\n=== GMAW Process — Existing Values Retained ===")
print(gmaw_process)

print("\n=== Derived Scalar Parameters for Rosenthal Model ===")

for key, value in process_scalars.items():
    print(f"  {key:30s}: {value}")


# ============================================================
# CORRECTED LAP-JOINT GEOMETRY OBJECT
# ============================================================
_geo = config["geometry"]

geometry = {
    "joint_type": _geo["joint_type"],
    "joint_configuration": _geo["joint_configuration"],
    "data_source": _geo["data_source"],
    "coordinate_system": _geo["coordinate_system"],

    "plate_length": Q_(
        _geo["plate_length"]["value"],
        _geo["plate_length"]["unit"],
    ),

    "plate_width": Q_(
        _geo["plate_width"]["value"],
        _geo["plate_width"]["unit"],
    ),

    "plate_thickness": Q_(
        _geo["plate_thickness"]["value"],
        _geo["plate_thickness"]["unit"],
    ),

    "assembly_length": Q_(
        _geo["assembly_length"]["value"],
        _geo["assembly_length"]["unit"],
    ),

    "assembly_width": Q_(
        _geo["assembly_width"]["value"],
        _geo["assembly_width"]["unit"],
    ),

    "lower_plate_length": Q_(
        _geo["lower_plate"]["length"]["value"],
        _geo["lower_plate"]["length"]["unit"],
    ),

    "lower_plate_width": Q_(
        _geo["lower_plate"]["width"]["value"],
        _geo["lower_plate"]["width"]["unit"],
    ),

    "lower_plate_thickness": Q_(
        _geo["lower_plate"]["thickness"]["value"],
        _geo["lower_plate"]["thickness"]["unit"],
    ),

    "lower_plate_x_range": Q_(
        _geo["lower_plate"]["x_range"]["values"],
        _geo["lower_plate"]["x_range"]["unit"],
    ),

    "lower_plate_y_range": Q_(
        _geo["lower_plate"]["y_range"]["values"],
        _geo["lower_plate"]["y_range"]["unit"],
    ),

    "lower_plate_z_range": Q_(
        _geo["lower_plate"]["z_range"]["values"],
        _geo["lower_plate"]["z_range"]["unit"],
    ),

    "upper_plate_length": Q_(
        _geo["upper_plate"]["length"]["value"],
        _geo["upper_plate"]["length"]["unit"],
    ),

    "upper_plate_width": Q_(
        _geo["upper_plate"]["width"]["value"],
        _geo["upper_plate"]["width"]["unit"],
    ),

    "upper_plate_thickness": Q_(
        _geo["upper_plate"]["thickness"]["value"],
        _geo["upper_plate"]["thickness"]["unit"],
    ),

    "upper_plate_x_range": Q_(
        _geo["upper_plate"]["x_range"]["values"],
        _geo["upper_plate"]["x_range"]["unit"],
    ),

    "upper_plate_y_range": Q_(
        _geo["upper_plate"]["y_range"]["values"],
        _geo["upper_plate"]["y_range"]["unit"],
    ),

    "upper_plate_z_range": Q_(
        _geo["upper_plate"]["z_range"]["values"],
        _geo["upper_plate"]["z_range"]["unit"],
    ),

    "overlap": Q_(
        _geo["overlap"]["value"],
        _geo["overlap"]["unit"],
    ),

    "overlap_y_range": Q_(
        _geo["overlap_y_range"]["values"],
        _geo["overlap_y_range"]["unit"],
    ),

    "fillet_leg": Q_(
        _geo["fillet_leg"]["value"],
        _geo["fillet_leg"]["unit"],
    ),

    "weld_count": _geo["weld_count"],
    "weld_sides": _geo["weld_sides"],

    "weld_length": Q_(
        _geo["weld_length"]["value"],
        _geo["weld_length"]["unit"],
    ),

    "weld_line_y_positions": Q_(
        _geo["weld_line_y_positions"]["values"],
        _geo["weld_line_y_positions"]["unit"],
    ),

    "weld_travel_direction": _geo["weld_travel_direction"],

    "weld_path_1_start": Q_(
        _geo["weld_path_1_start"]["values"],
        _geo["weld_path_1_start"]["unit"],
    ),

    "weld_path_1_end": Q_(
        _geo["weld_path_1_end"]["values"],
        _geo["weld_path_1_end"]["unit"],
    ),

    "weld_path_2_start": Q_(
        _geo["weld_path_2_start"]["values"],
        _geo["weld_path_2_start"]["unit"],
    ),

    "weld_path_2_end": Q_(
        _geo["weld_path_2_end"]["values"],
        _geo["weld_path_2_end"]["unit"],
    ),

    "thermo_distance": Q_(
        _geo["sensor_distances"]["values"],
        _geo["sensor_distances"]["unit"],
    ),

    "sensor_labels": _geo["sensor_labels"],

    "thermocouple_reference_weld": (
        _geo["thermocouple_reference_weld"]
    ),

    "thermocouple_x_positions": Q_(
        _geo["thermocouple_x_positions"]["values"],
        _geo["thermocouple_x_positions"]["unit"],
    ),

    "thermocouple_y_positions": Q_(
        _geo["thermocouple_y_positions"]["values"],
        _geo["thermocouple_y_positions"]["unit"],
    ),

    "thermocouple_z_positions": Q_(
        _geo["thermocouple_z_positions"]["values"],
        _geo["thermocouple_z_positions"]["unit"],
    ),
}

print("\n=== Corrected Lap-Joint Geometry ===")

for key, value in geometry.items():
    print(f"  {key:30s}: {value}")


# ============================================================
# CREATE WELDX CORE FILE
# ============================================================
core_path = Path(CORE_FILE)
core_path.unlink(missing_ok=True)

project_info = {
    "date_created": _created_at,
    "weldx_version": _weldx_version,
    "numpy_version": _numpy_version,
    "description": (
        "Analytical Rosenthal 3D heat-source model with Monte Carlo "
        "uncertainty propagation for GMAW of an Aluminium 6061-T6 "
        "double-fillet lap joint. Both plates are 200 mm long, 80 mm "
        "wide and 4 mm thick. The plates overlap by 25 mm across their "
        "width. Both fillet welds run along the full 200 mm plate length. "
        "Thermocouples are located 2 mm, 5 mm and 10 mm from Weld 1. "
        "Existing GMAW and Monte Carlo settings are retained."
    ),
    "standard_references": [
        "AL 6061-T6",
        "ASTM B209",
    ],
}

with WeldxFile(CORE_FILE, mode="rw") as wx:
    wx["project_info"] = project_info
    wx["material"] = material
    wx["gmaw_process"] = gmaw_process
    wx["process"] = process_scalars
    wx["geometry"] = geometry

    if _monte_carlo_config_unchanged is not None:
        wx["monte_carlo"] = _monte_carlo_config_unchanged

print(f"\nCore file saved: {CORE_FILE}")
print(f"File size: {core_path.stat().st_size} bytes")


# ============================================================
# RELOAD AND VERIFY
# ============================================================
with WeldxFile(CORE_FILE, mode="r") as wx:
    loaded = wx.copy()

print("\nMODEL VERIFICATION SUMMARY")

print("\nMaterial:")
print(f"  Material          : {loaded['material']['common_name']}")
print(f"  Standard          : {loaded['material']['standard']}")
print(f"  Density           : {loaded['material']['density']}")
print(f"  Conductivity      : {loaded['material']['thermal_conductivity']}")

print("\nProcess:")
print(f"  Process Type      : {loaded['gmaw_process'].base_process}")
print(f"  Voltage           : {loaded['gmaw_process'].parameters['voltage'].data}")
print(f"  Current           : {loaded['gmaw_process'].parameters['current'].data}")
print(
    f"  Wire Feed Rate    : "
    f"{loaded['gmaw_process'].parameters['wire_feedrate'].data}"
)

print("\nGeometry:")
print(f"  Joint Type        : {loaded['geometry']['joint_type']}")
print(f"  Plate Length      : {loaded['geometry']['plate_length']}")
print(f"  Plate Width       : {loaded['geometry']['plate_width']}")
print(f"  Plate Thickness   : {loaded['geometry']['plate_thickness']}")
print(f"  Assembly Width    : {loaded['geometry']['assembly_width']}")
print(f"  Overlap           : {loaded['geometry']['overlap']}")
print(f"  Fillet Leg        : {loaded['geometry']['fillet_leg']}")
print(f"  Number of Welds   : {loaded['geometry']['weld_count']}")
print(f"  Weld Length       : {loaded['geometry']['weld_length']}")
print(f"  Weld Y Positions  : {loaded['geometry']['weld_line_y_positions']}")
print(f"  Weld Direction    : {loaded['geometry']['weld_travel_direction']}")
print(f"  Thermocouples     : {loaded['geometry']['sensor_labels']}")
print(f"  TC Distances      : {loaded['geometry']['thermo_distance']}")
print(f"  TC X Positions    : {loaded['geometry']['thermocouple_x_positions']}")
print(f"  TC Y Positions    : {loaded['geometry']['thermocouple_y_positions']}")
print(f"  TC Z Positions    : {loaded['geometry']['thermocouple_z_positions']}")

print("\nMonte Carlo:")

if "monte_carlo" in loaded:
    print("  Monte Carlo configuration retained")
else:
    print(
        "  No Monte Carlo section was present "
        "in the original config.yaml"
    )

print("\nStatus:")
print(
    "  Corrected Al6061 double-fillet lap-joint core model "
    "loaded successfully"
)
print(
    "  Plate length and both weld paths are 200 mm; "
    "plate width is 80 mm."
)

Initializing WelDX core file: project_core.asdf
Loaded and updated configuration: config.yaml
  Material       : Aluminium 6061-T6
  Process        : GMAW
  Voltage/Current: 22.0 V / 180.0 A
  Wire feed      : 7.0 m/min
  Plate size     : 200 × 80 × 4 mm
  Overlap        : 25 mm across plate width
  Weld length    : 200 mm per fillet weld
  Weld direction : +x

=== Aluminium 6061-T6 Material Properties ===
  common_name                   : Aluminium 6061-T6
  standard                      : AL 6061-T6 / ASTM B209
  density                       : 2700.0 kg / m ** 3
  thermal_conductivity          : 167.0 W / K / m
  specific_heat                 : 896.0 J / K / kg
  melting_point                 : 652.0 °C
  solidus_temperature           : 582.0 °C
  ambient_temperature           : 293.15 K
  data_source                   : Representative room-temperature thermophysical properties for wrought Aluminium 6061-T6.
  thermal_diffusivity           : 6.903108465608466e-05 m ** 2 / s

=== GMA